# Day10 · Xuất hình minh họa và kiểm chứng tọa độ
**Không training, không tải model, không cần GPU.** Chọn Accelerator=None.

1. Add Input dataset road-damage như các notebook trước.
2. Nếu còn session training: dùng trực tiếp `/kaggle/working/day10/work/e3_test.jsonl`.
3. Nếu session mới: upload `day10_results (7).zip` thành Kaggle Dataset **private**, rồi Add Input dataset đó. Có thể upload riêng `e3_test.jsonl` thay ZIP.
4. Chạy tuần tự, tải `day10_visuals.zip` về local. Notebook tự tìm predictions E3 và chuẩn bị split bằng script gốc.

Ảnh được chọn có chủ đích để giải thích TP/FP/FN, không đại diện cho tỉ lệ lỗi toàn bộ test. Dùng kết quả E3 300 bước (F1 khoảng 0,4599) để khớp báo cáo chính thức.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, zipfile
REPO_URL = "https://github.com/thanhhuyvan/K4-Track-4-Day10-VanThanhHuy-2A202602763--VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
# Có thể trỏ đến repo được upload dưới dạng Kaggle Input nếu GitHub chưa có mã đề.
REPO_INPUT = None
DATA_INPUT = None  # None: tìm tự động trong /kaggle/input; hoặc nhập đường dẫn dataset
LOAD_MODEL = False  # đổi True khi đã bật GPU và sẵn sàng tải model

def run(*args):
    subprocess.run([str(x) for x in args], check=True, cwd=ROOT if ROOT.exists() else None)

if not ROOT.exists():
    if REPO_INPUT:
        shutil.copytree(REPO_INPUT, ROOT)
    else:
        run("git", "clone", REPO_URL, ROOT)
for required in ["scripts/prepare_data.py", "scripts/score.py", "requirements-colab.txt"]:
    assert (ROOT / required).is_file(), f"Repo thiếu {required}; upload repo vào Kaggle Input và đặt REPO_INPUT."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
(ROOT / "work").mkdir(exist_ok=True)
(ROOT / "report/evidence").mkdir(parents=True, exist_ok=True)
# Ghi hash để phát hiện vô tình sửa công cụ đề.
PROTECTED = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
             for folder in ["scripts", "tests"] for p in (ROOT / folder).rglob("*.py")}
print("Repo:", ROOT)


## 1. Dependencies nhẹ và split chuẩn

In [ ]:
run(sys.executable,"-m","pip","install","pillow","matplotlib")


In [ ]:
from scripts.prepare_data import _find_root
base = Path(DATA_INPUT) if DATA_INPUT else Path("/kaggle/input")
try:
    source = _find_root(base)
except FileNotFoundError:
    raise RuntimeError("Chưa tìm thấy images/ và labels-YOLO/. Add Input dataset hoặc đặt DATA_INPUT đúng thư mục.")
run(sys.executable, "scripts/prepare_data.py", "--source", source)
meta = json.loads((ROOT / "data/splits/meta.json").read_text())
splits = {s: [json.loads(line) for line in (ROOT / f"data/splits/{s}.jsonl").read_text().splitlines() if line.strip()]
          for s in ["train", "val", "test"]}
sessions = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not sessions["train"] & sessions["val"]
assert not sessions["train"] & sessions["test"]
assert not sessions["val"] & sessions["test"]
print(json.dumps(meta["counts"], indent=2))
print("dataset_version:", meta["dataset_version"], "— không trùng phiên quay")


## 2. Tìm predictions — không đọc trọng số hoặc giải nén toàn bộ ZIP

In [ ]:
from pathlib import Path
import zipfile, json, hashlib
# Đặt đường dẫn cụ thể nếu có nhiều gói kết quả E3 khác nhau.
PRED_SOURCE = None
# Ví dụ: "/kaggle/input/your-private-results/day10_results (7).zip"

def read_prediction_source(path):
    path=Path(path)
    if path.suffix.lower()==".zip":
        with zipfile.ZipFile(path) as z:
            matches=[name for name in z.namelist() if name.endswith("work/e3_test.jsonl")]
            if len(matches)!=1: raise ValueError(f"ZIP không có đúng 1 E3 predictions: {path}")
            return z.read(matches[0])
    return path.read_bytes()

if PRED_SOURCE:
    chosen=Path(PRED_SOURCE); payload=read_prediction_source(chosen)
else:
    candidates=[]
    local=ROOT/"work/e3_test.jsonl"
    if local.exists(): candidates.append(local)
    candidates+=sorted(Path("/kaggle/input").rglob("e3_test.jsonl"))
    for p in sorted(Path("/kaggle/input").rglob("*.zip")):
        try:
            with zipfile.ZipFile(p) as z:
                if any(name.endswith("work/e3_test.jsonl") for name in z.namelist()): candidates.append(p)
        except zipfile.BadZipFile: pass
    assert candidates,"Chưa có E3 predictions. Add Input ZIP kết quả (7) hoặc e3_test.jsonl."
    variants={}
    for p in candidates:
        data=read_prediction_source(p)
        variants.setdefault(hashlib.sha256(data).hexdigest(),[]).append((p,data))
    assert len(variants)==1,"Có nhiều bản E3 khác nhau. Đặt PRED_SOURCE trỏ đúng ZIP (7)/E3 300 bước."
    chosen,payload=next(iter(variants.values()))[0]
pred_path=ROOT/"work/visual_e3_test.jsonl"
pred_path.write_bytes(payload)
rows=[json.loads(line) for line in payload.decode("utf-8").splitlines() if line.strip()]
assert len(rows)==len({r["image_id"] for r in rows})==len(splits["test"])
assert {r["image_id"] for r in rows}=={r["image_id"] for r in splits["test"]}
predictions={r["image_id"]:r["raw"] for r in rows}
print("Nguồn predictions:",chosen)
print("SHA256:",hashlib.sha256(payload).hexdigest())


## 3. Chấm kiểm chứng norm1000 và sai chế độ pixel bằng scorer gốc

In [ ]:
diagnostics=ROOT/"report/diagnostics";diagnostics.mkdir(parents=True,exist_ok=True)
for coord in ["norm1000","pixel"]:
    run(sys.executable,"scripts/score.py","--pred",pred_path,"--split","test",
        "--coord",coord,"--name",f"E3_coordinate_check_{coord}",
        "--out",diagnostics/f"e3_coord_{coord}.json")
correct=json.loads((diagnostics/"e3_coord_norm1000.json").read_text())
wrong=json.loads((diagnostics/"e3_coord_pixel.json").read_text())
# Không đổi lựa chọn model: xác nhận đúng predictions đã có trong báo cáo.
assert abs(correct["micro"]["f1"]-0.4598737601442741)<1e-10, "Đây không phải predictions E3 300 bước mới. Chọn lại PRED_SOURCE."
print("F1 norm1000:",correct["micro"]["f1"],"F1 chấm sai pixel:",wrong["micro"]["f1"])


## 4. Ghép box cùng lớp, IoU ≥0,5 và chọn 6 ảnh giải thích

In [ ]:
from scripts.score import parse_output, iou_xyxy, _gt_boxes

def analyze(row):
    gt=_gt_boxes(row)
    pred=parse_output(predictions[row["image_id"]])
    assert pred is not None,"E3 hiện tại dự kiến có 100% output hợp lệ"
    used_g=set();used_p=set();pairs=[]
    for label in sorted({b["label"] for b in gt+pred}):
        gs=[i for i,b in enumerate(gt) if b["label"]==label]
        ps=[j for j,b in enumerate(pred) if b["label"]==label]
        options=sorted([(iou_xyxy(gt[i]["bbox"],pred[j]["bbox"]),i,j) for i in gs for j in ps],reverse=True)
        for value,i,j in options:
            if value<.5: break
            if i in used_g or j in used_p:continue
            used_g.add(i);used_p.add(j);pairs.append((i,j,value))
    return dict(row=row,gt=gt,pred=pred,pairs=pairs,matched_g=used_g,matched_p=used_p,
                tp=len(pairs),fp=len(pred)-len(used_p),fn=len(gt)-len(used_g))

analyses=[analyze(r) for r in sorted(splits["test"],key=lambda r:r["image_id"])]
assert sum(a["tp"] for a in analyses)==correct["micro"]["tp"]
assert sum(a["fp"] for a in analyses)==correct["micro"]["fp"]
assert sum(a["fn"] for a in analyses)==correct["micro"]["fn"]
selected=[];used=set()
def pick(category,predicate,key):
    options=[a for a in analyses if a["row"]["image_id"] not in used and predicate(a)]
    assert options,f"Không tìm được ảnh cho {category}"
    a=sorted(options,key=key)[0];selected.append((category,a));used.add(a["row"]["image_id"])

for label in ["manhole","pothole","crack"]:
    pick("TP_"+label,
        lambda a,l=label:any(a["gt"][i]["label"]==l for i,j,v in a["pairs"]),
        lambda a:(a["fp"]+a["fn"],-a["tp"],a["row"]["image_id"]))
pick("False_positive",lambda a:a["fp"]>0,lambda a:(-a["fp"],a["row"]["image_id"]))
pick("False_negative",lambda a:a["fn"]>0,lambda a:(-a["fn"],a["row"]["image_id"]))
# Khó phân biệt: unmatched prediction gần một GT nhưng chưa đạt đúng lớp/IoU.
def near_miss(a):
    return any(iou_xyxy(p["bbox"],g["bbox"])>.1
               for j,p in enumerate(a["pred"]) if j not in a["matched_p"] for g in a["gt"])
pick("Localization_or_class_error",near_miss,lambda a:(a["row"]["image_id"],))
for category,a in selected:print(category,a["row"]["image_id"],"TP/FP/FN",a["tp"],a["fp"],a["fn"])


## 5. Vẽ GT xanh và prediction đỏ — lưu từng ảnh cùng bảng chú thích

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
images_dir=ROOT/"report/images";images_dir.mkdir(parents=True,exist_ok=True)
manifest=[]
for index,(category,a) in enumerate(selected,1):
    row=a["row"]
    with Image.open(ROOT/"data"/row["image_path"]) as im:image=im.convert("RGB")
    fig,ax=plt.subplots(figsize=(14,9));ax.imshow(image)
    for kind,boxes,color,matched in [("GT",a["gt"],"lime",a["matched_g"]),("PRED",a["pred"],"red",a["matched_p"])]:
        for k,b in enumerate(boxes):
            x1,y1,x2,y2=b["bbox"]
            x1,x2=x1*image.width/1000,x2*image.width/1000
            y1,y2=y1*image.height/1000,y2*image.height/1000
            status="TP" if k in matched else "FN" if kind=="GT" else "FP"
            ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor=color,
                linewidth=2,linestyle="--" if kind=="GT" else "-"))
            y=y1-3 if kind=="GT" else y2+9
            ax.text(x1,y,f"{kind} {b['label']} {status}",fontsize=7,color=color,
                bbox=dict(facecolor="black",alpha=.65,pad=1))
    ax.set_title(f"{category} | {row['image_id']}\nGT: green dashed · E3: red solid | TP={a['tp']} FP={a['fp']} FN={a['fn']}",fontsize=11)
    ax.set_xlim(0,image.width);ax.set_ylim(image.height+22,-15);ax.axis("off")
    filename=f"{index:02d}_{category}.png"
    fig.tight_layout();fig.savefig(images_dir/filename,dpi=160,bbox_inches="tight")
    plt.show();plt.close(fig)
    manifest.append(dict(file=filename,category=category,image_id=row["image_id"],tp=a["tp"],fp=a["fp"],fn=a["fn"],
        gt=a["gt"],pred=a["pred"],matches=a["pairs"]))
(images_dir/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
captions=["# E3 prediction illustrations", "", "GT: green dashed. Prediction: red solid. TP requires same class and IoU >= 0.5.","", "Purposively selected examples for explanation; not a random sample or a new performance estimate.",""]
for item in manifest:
    captions.extend([f"## {item['category']} — {item['image_id']}","",f"TP={item['tp']}; FP={item['fp']}; FN={item['fn']}.","",f"![{item['category']}]({item['file']})",""])
(images_dir/"README.md").write_text("\n".join(captions),encoding="utf-8")


## 6. Tải gói hình và diagnostics về local

In [ ]:
from IPython.display import FileLink,display
archive=Path("/kaggle/working/day10_visuals.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for folder in [images_dir,diagnostics]:
        for p in folder.rglob("*"):
            if p.is_file():z.write(p,str(p.relative_to(ROOT)))
    provenance={"prediction_source":str(chosen),"prediction_sha256":hashlib.sha256(payload).hexdigest(),
        "dataset_version":meta["dataset_version"],"experiment":"E3_300_steps", "selection":"purposive TP per class + FP + FN + near miss"}
    z.writestr("report/diagnostics/visual_provenance.json",json.dumps(provenance,indent=2))
print(archive,"MiB:",archive.stat().st_size/2**20)
display(FileLink(str(archive)))
